In [1]:
import sys
import os
import winreg
from pathlib import Path

# 检查 Python，避免新 Notebook 使用系统默认解释器。
if sys.version_info[:3] != (3, 13, 16):
    raise RuntimeError("请选择项目 .venv 的 Python 3.13.16 内核。")

# 从 Windows 配置读取真实桌面位置，不依赖 Notebook 工作目录。
registry_path = (
    r"Software\Microsoft\Windows\CurrentVersion"
    r"\Explorer\User Shell Folders"
)

with winreg.OpenKey(winreg.HKEY_CURRENT_USER, registry_path) as key:
    configured_desktop, _ = winreg.QueryValueEx(key, "Desktop")

desktop = Path(os.path.expandvars(configured_desktop))
PROJECT_ROOT = desktop / "tft-balance-simulator"

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("未找到桌面项目，请检查桌面配置。")

# 确保当前内核属于这个项目的虚拟环境。
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("请切换到桌面项目的 .venv 内核。")

print("Python 版本：", sys.version.split()[0])
print("项目定位与虚拟环境检查通过")

Python 版本： 3.13.16
项目定位与虚拟环境检查通过


找到下载的TFT Set 10 JSON文件并且读取

In [5]:
import json
import hashlib
from collections import Counter

import pandas as pd
import numpy as np
from IPython.display import display

# 从已保存的快照读取
snapshot_dir = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "set10_patch14_1"
    / "20261027T211906309960Z_ae6d4bde7f59"
)

manifest = json.loads(
    (snapshot_dir / "manifest.json").read_text(encoding="utf-8")
)
snapshot_bytes = (snapshot_dir / "en_us.json").read_bytes()

actual_sha256 = hashlib.sha256(snapshot_bytes).hexdigest()

if actual_sha256 != manifest.get("sha256"):
    raise ValueError("原始文件与来源记录中的哈希不一致。")

if (
    manifest.get("set_number") != 10
    or manifest.get("patch_directory") != "14.1"
):
    raise ValueError("快照不是本次实验选定的 Set 10 / Patch 14.1。")

snapshot_data = json.loads(snapshot_bytes)
source_units = snapshot_data["sets"]["10"]["champions"]

print("快照验证通过")
print("原始单位记录数：", len(source_units))
print("pandas 版本：", pd.__version__)

快照验证通过
原始单位记录数： 76
pandas 版本： 3.1.0
